#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length

#Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_sales_details")

#Silver Transformations

## Trim leading & trailing spaces

In [0]:
#Instead of reconstructing the DataFrame plan for each column using a for loop, this method updates all string columns in a single plan, making it faster.
#.alias(file.name) - keeps the origianl name of the column instead of "trim(prd_id)" for example

# Old Code:
# for field in df.schema.fields:
#     if isinstance(field.dataType, StringType):
#         df = df.withColumn(field.name, trim(col(field.name)))


df = df.select([
    trim(col(field.name)).alias(field.name) if isinstance(field.dataType, StringType) 
    else col(field.name) 
    for field in df.schema.fields
])

#Cleaning Dates

In [0]:
# Replaces incorrect entries (zeros or values that are not precisely eight characters long) with nulls in the sales order, shipping, and due date columns. 
# Next, valid 8-digit values are transformed into correctly structured calendar dates (YYYY-MM-DD) from raw text or integers.
#I could check to see if order date is less than shipping date and due date is more than shipping date, but that's not necessary.


df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

#Sales & Price Corrections

In [0]:
# Fixes missing, zero, or negative sales prices by back-calculating them: total sales divided by quantity (Sales price = sales / quantity).
# If the quantity is zero, it assigns a null value to prevent a division-by-zero error, while keeping valid existing prices unchanged 


In [0]:
df = (
    df
    # 1. Recalculate sls_sales based on SQL logic
    .withColumn(
        "sls_sales",
        F.when(
            col("sls_sales").isNull() | 
            (col("sls_sales") <= 0) | 
            (col("sls_sales") != (col("sls_quantity") * F.abs(col("sls_price")))), #comma is important to separate the bad conditions above
            col("sls_quantity") * F.abs(col("sls_price")) #If any of the bad data checks above are true, set sls_sales to sls_quantity * sls_price
        ).otherwise(col("sls_sales")) #If there's no need for a change keep sls_sales as it is
    )
    # 2. Recalculate sls_price based on SQL logic
    .withColumn(
        "sls_price",
        F.when(
            col("sls_price").isNull() | (col("sls_price") <= 0), #When sales price is NULL or less than or equal to zero
            col("sls_sales") / F.nullif(col("sls_quantity"), F.lit(0)) #Divide sales with quantity to determine price & to prevent ABS error set quantity to null if it's zero
        ).otherwise(col("sls_price")) #If there's no need for a change keep sls_price as it is
    )
)

#Renaming Columns

In [0]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#Sanity checks of dataframe

In [0]:
df.limit(10).display()

#Write Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_sales")

#Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 30